In [4]:
import pandas as pd
import numpy as np
import os

BASE_DIR = ""
RAW_PATH = f"{BASE_DIR}/data/raw/emdat_raw.xlsx"
INTERIM_PATH = f"{BASE_DIR}/data/interim/"

# ── ISO3 standardization map ───────────────────────────────────────────────
# Ensures consistent ISO3 codes across all datasets
ISO3_MAP = {
    "BLZ": "BLZ",  # Belize
    "COL": "COL",  # Colombia
    "CRI": "CRI",  # Costa Rica
    "CUB": "CUB",  # Cuba
    "DOM": "DOM",  # Dominican Republic
    "SLV": "SLV",  # El Salvador
    "GTM": "GTM",  # Guatemala
    "GUY": "GUY",  # Guyana
    "HTI": "HTI",  # Haiti
    "HND": "HND",  # Honduras
    "JAM": "JAM",  # Jamaica
    "NIC": "NIC",  # Nicaragua
    "PAN": "PAN",  # Panama
    "PRI": "PRI",  # Puerto Rico
    "SUR": "SUR",  # Suriname
    "TTO": "TTO",  # Trinidad and Tobago
    "VEN": "VEN",  # Venezuela
    "BHS": "BHS",  # Bahamas
    "BRB": "BRB",  # Barbados
}

# Canonical country name map — standardizes long/variant names
COUNTRY_NAME_MAP = {
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "Trinidad and Tobago": "Trinidad and Tobago",
    "Dominican Republic": "Dominican Republic",
}

# Role classification — optimization targets vs pressure inputs
COUNTRY_ROLE = {
    "BLZ": "tier1", "CRI": "tier1", "CUB": "tier1", "DOM": "tier1",
    "SLV": "tier1", "GTM": "tier1", "GUY": "tier1", "HTI": "tier1",
    "HND": "tier1", "JAM": "tier1", "NIC": "tier1", "PAN": "tier1",
    "PRI": "tier1", "SUR": "tier1", "TTO": "tier1", "BHS": "tier1",
    "BRB": "tier1",
    "COL": "pressure_input",  # transit corridor
    "VEN": "pressure_input",  # displacement origin
}

# Disaster type taxonomy — maps EM-DAT types to project categories
DISASTER_TYPE_MAP = {
    "Flood": "flood",
    "Storm": "storm",
    "Earthquake": "earthquake",
    "Drought": "drought",
    "Extreme temperature": "heatwave",
}


def load_raw(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    print(f"  Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
    return df


def select_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only columns relevant to the project."""
    keep = [
        "DisNo.", "ISO", "Country",
        "Disaster Type", "Disaster Subtype",
        "Associated Types",
        "Start Year", "Start Month", "Start Day",
        "End Year", "End Month", "End Day",
        "Total Deaths", "No. Injured",
        "No. Affected", "No. Homeless", "Total Affected",
        "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)",
        "AID Contribution ('000 US$)",
        "Event Name", "Location",
    ]
    df = df[keep].copy()
    print(f"  Columns reduced to: {df.shape[1]}")
    return df


def standardize_identifiers(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize country names, ISO codes, and add role column."""
    # Clean country names
    df["Country"] = df["Country"].replace(COUNTRY_NAME_MAP)

    # Standardize ISO3 — remap Venezuela ISO if needed
    df["ISO3"] = df["ISO"].str.upper().str.strip()

    # Add country role
    df["country_role"] = df["ISO3"].map(COUNTRY_ROLE).fillna("unknown")

    # Flag unknowns
    unknown = df[df["country_role"] == "unknown"]["ISO3"].unique()
    if len(unknown) > 0:
        print(f"  ⚠️  Unknown country roles: {unknown}")

    return df


def standardize_disaster_types(df: pd.DataFrame) -> pd.DataFrame:
    """Map EM-DAT disaster types to project taxonomy."""
    df["disaster_category"] = df["Disaster Type"].map(DISASTER_TYPE_MAP)

    # Log unmapped types
    unmapped = df[df["disaster_category"].isna()]["Disaster Type"].unique()
    if len(unmapped) > 0:
        print(f"  ⚠️  Unmapped disaster types: {unmapped}")

    return df


def build_date_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build clean year/month fields.
    Start Month is fully populated (confirmed in inspection).
    Use Start Year + Start Month as the event's country-month key.
    """
    df["event_year"] = df["Start Year"].astype(int)
    df["event_month"] = df["Start Month"].astype(int)

    # Validate range
    assert df["event_year"].between(2019, 2024).all(), "Year out of range"
    assert df["event_month"].between(1, 12).all(), "Month out of range"

    # Build year-month string key for joins
    df["year_month"] = (
        df["event_year"].astype(str) + "-" +
        df["event_month"].astype(str).str.zfill(2)
    )

    return df


def build_severity_score(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construct a composite severity score per event (0–1 normalized).

    Components:
      - Total Affected (86% coverage)  — weight 0.5
      - Total Deaths   (62% coverage)  — weight 0.5

    Total Damage excluded — only 21% coverage, unreliable.

    Each component is log-scaled before normalization to handle
    the extreme skew in disaster impact data.
    """
    # Log-scale to compress extreme outliers (add 1 to handle zeros)
    df["log_affected"] = np.log1p(df["Total Affected"].fillna(0))
    df["log_deaths"] = np.log1p(df["Total Deaths"].fillna(0))

    # Min-max normalize each component across all events
    def minmax(series):
        mn, mx = series.min(), series.max()
        if mx == mn:
            return series * 0
        return (series - mn) / (mx - mn)

    df["norm_affected"] = minmax(df["log_affected"])
    df["norm_deaths"] = minmax(df["log_deaths"])

    # Weighted composite
    df["severity_score"] = (
        0.5 * df["norm_affected"] +
        0.5 * df["norm_deaths"]
    ).round(4)

    # Flag events with zero data (both affected and deaths are null/zero)
    df["severity_data_flag"] = np.where(
        (df["Total Affected"].isna()) & (df["Total Deaths"].isna()),
        "no_impact_data",
        "ok"
    )

    no_data = (df["severity_data_flag"] == "no_impact_data").sum()
    print(f"  Events with no impact data (affected + deaths both null): {no_data}")

    return df


def aggregate_to_country_month(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate event-level data to country-month.

    For each country-month:
      - event_count         : number of disaster events
      - max_severity        : peak severity event that month
      - sum_affected        : total people affected
      - sum_deaths          : total deaths
      - disaster_types      : pipe-separated list of disaster categories
      - has_storm           : binary flag
      - has_flood           : binary flag
      - has_earthquake      : binary flag
      - has_drought         : binary flag
      - compound_event      : 1 if >1 disaster type occurred same month
    """
    agg = df.groupby(["ISO3", "Country", "country_role", "year_month",
                      "event_year", "event_month"]).agg(
        event_count=("DisNo.", "count"),
        max_severity=("severity_score", "max"),
        mean_severity=("severity_score", "mean"),
        sum_affected=("Total Affected", "sum"),
        sum_deaths=("Total Deaths", "sum"),
        disaster_types=("disaster_category", lambda x: "|".join(sorted(x.dropna().unique()))),
    ).reset_index()

    # Binary disaster type flags
    agg["has_storm"] = agg["disaster_types"].str.contains("storm").astype(int)
    agg["has_flood"] = agg["disaster_types"].str.contains("flood").astype(int)
    agg["has_earthquake"] = agg["disaster_types"].str.contains("earthquake").astype(int)
    agg["has_drought"] = agg["disaster_types"].str.contains("drought").astype(int)
    agg["has_heatwave"] = agg["disaster_types"].str.contains("heatwave").astype(int)

    # Compound event flag — multiple disaster types in same country-month
    type_count = (
        agg["has_storm"] + agg["has_flood"] +
        agg["has_earthquake"] + agg["has_drought"] + agg["has_heatwave"]
    )
    agg["compound_event"] = (type_count > 1).astype(int)

    # Sort
    agg = agg.sort_values(["ISO3", "event_year", "event_month"]).reset_index(drop=True)

    print(f"  Country-month records: {len(agg)}")
    print(f"  Compound events: {agg['compound_event'].sum()}")
    print(f"  Countries: {agg['ISO3'].nunique()}")

    return agg


def validate(df_events: pd.DataFrame, df_cm: pd.DataFrame):
    """Basic validation checks."""
    print("\n── Validation ──────────────────────────────")

    # Check year range
    assert df_events["event_year"].between(2019, 2024).all()
    print("  ✅ Year range: 2019–2024")

    # Check all countries accounted for
    expected_isos = set(COUNTRY_ROLE.keys())
    actual_isos = set(df_cm["ISO3"].unique())
    missing = expected_isos - actual_isos
    if missing:
        print(f"  ⚠️  Countries with no disaster events: {missing}")
        print(f"     (These will appear as zeros in the master grid — expected)")
    else:
        print("  ✅ All expected countries present")

    # Severity score range
    assert df_events["severity_score"].between(0, 1).all()
    print("  ✅ Severity scores: 0–1 range confirmed")

    # No duplicate DisNo.
    assert df_events["DisNo."].nunique() == len(df_events)
    print("  ✅ No duplicate disaster IDs")

    print("────────────────────────────────────────────\n")


def main():
    print("\n🔄 EM-DAT Cleaning Pipeline")
    print("=" * 45)

    print("\n[1/7] Loading raw data...")
    df = load_raw(RAW_PATH)

    print("\n[2/7] Selecting columns...")
    df = select_columns(df)

    print("\n[3/7] Standardizing identifiers...")
    df = standardize_identifiers(df)

    print("\n[4/7] Standardizing disaster types...")
    df = standardize_disaster_types(df)

    print("\n[5/7] Building date fields...")
    df = build_date_fields(df)

    print("\n[6/7] Building severity scores...")
    df = build_severity_score(df)

    print("\n[7/7] Aggregating to country-month...")
    df_cm = aggregate_to_country_month(df)

    # Validate
    validate(df, df_cm)

    # Save outputs
    event_path = os.path.join(INTERIM_PATH, "emdat_clean.csv")
    cm_path = os.path.join(INTERIM_PATH, "emdat_country_month.csv")

    df.to_csv(event_path, index=False)
    df_cm.to_csv(cm_path, index=False)

    # Summary
    print("\n── Output Summary ──────────────────────────")
    print(f"  emdat_clean.csv        : {len(df)} events")
    print(f"  emdat_country_month.csv: {len(df_cm)} country-month records")
    print(f"  Year range             : {df['event_year'].min()}–{df['event_year'].max()}")
    print(f"  Countries              : {df['ISO3'].nunique()}")
    print(f"  Severity score range   : {df['severity_score'].min():.3f} – {df['severity_score'].max():.3f}")
    print("────────────────────────────────────────────\n")

    return df, df_cm


if __name__ == "__main__":
    main()


🔄 EM-DAT Cleaning Pipeline

[1/7] Loading raw data...
  Loaded: 169 rows × 47 columns

[2/7] Selecting columns...
  Columns reduced to: 22

[3/7] Standardizing identifiers...

[4/7] Standardizing disaster types...

[5/7] Building date fields...

[6/7] Building severity scores...
  Events with no impact data (affected + deaths both null): 10

[7/7] Aggregating to country-month...
  Country-month records: 142
  Compound events: 11
  Countries: 19

── Validation ──────────────────────────────
  ✅ Year range: 2019–2024
  ✅ All expected countries present
  ✅ Severity scores: 0–1 range confirmed
  ✅ No duplicate disaster IDs
────────────────────────────────────────────


── Output Summary ──────────────────────────
  emdat_clean.csv        : 169 events
  emdat_country_month.csv: 142 country-month records
  Year range             : 2019–2024
  Countries              : 19
  Severity score range   : 0.000 – 0.939
────────────────────────────────────────────

